In [ ]:
'''
predicting actions that correspond to input timesteps.
'''
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import time
import math
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1.1 Import Data

In [ ]:
import os
import h5py
import matplotlib.pyplot as plt
'''
# Training:
input: [qpos_t0, qpos_t1, ..., qpos_t99]     # 100 position timesteps
target: [action_t0, action_t1, ..., action_t99] # 100 action timesteps

# The model learns: "Given 100 position timesteps, predict 100 corresponding actions"
'''

sequence_length = 100 # number of input steps
output_window = 1 # # This is misleading -  actually predict 100 steps!
batch_size = 250

def load_hdf5(dataset_path):
    if not os.path.isfile(dataset_path):
        print(f'Dataset does not exist at \n{dataset_path}\n')
        exit()

    with h5py.File(dataset_path, 'r') as root:
        qpos = root['/observations/qpos'][()]
        qvel = root['/observations/qvel'][()]
        image_dict = dict()
        for cam_name in root[f'/observations/images/'].keys():
            image_dict[cam_name] = root[f'/observations/images/{cam_name}'][()]
        action = root['/action'][()]

    return qpos, qvel, action, image_dict

In [ ]:
import os
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def load_hdf5(dataset_path):
    if not os.path.isfile(dataset_path):
        print(f'Dataset does not exist at \n{dataset_path}\n')
        return None, None, None, None

    with h5py.File(dataset_path, 'r') as root:
        qpos = root['/observations/qpos'][()]
        qvel = root['/observations/qvel'][()]
        image_dict = dict()
        for cam_name in root[f'/observations/images/'].keys():
            image_dict[cam_name] = root[f'/observations/images/{cam_name}'][()]
        action = root['/action'][()]

    return qpos, qvel, action, image_dict

def load_all_episodes(data_dir):
    """
    Load all HDF5 files from the specified directory
    Args:
        data_dir: directory containing HDF5 files
    Returns:
        combined qpos, qvel, action arrays and list of image dicts
    """
    print(f"Loading episodes from: {data_dir}")
    
    # Find all HDF5 files
    hdf5_files = [f for f in os.listdir(data_dir) if f.endswith('.hdf5')]
    hdf5_files.sort()  # Sort for consistent ordering
    
    if not hdf5_files:
        print("No HDF5 files found in the directory!")
        return None, None, None, None
    
    print(f"Found {len(hdf5_files)} HDF5 files: {hdf5_files}")
    
    all_qpos = []
    all_qvel = []
    all_action = []
    all_image_dicts = []
    
    episode_info = []
    
    for filename in hdf5_files:
        filepath = os.path.join(data_dir, filename)
        print(f"Loading {filename}...")
        
        qpos, qvel, action, image_dict = load_hdf5(filepath)
        
        if qpos is not None:
            print(f"  - Episode length: {len(qpos)} timesteps")
            print(f"  - QPos shape: {qpos.shape}")
            print(f"  - Action shape: {action.shape}")
            
            all_qpos.append(qpos)
            all_qvel.append(qvel)
            all_action.append(action)
            all_image_dicts.append(image_dict)
            
            episode_info.append({
                'filename': filename,
                'length': len(qpos),
                'start_idx': sum(len(ep) for ep in all_qpos[:-1]),
                'end_idx': sum(len(ep) for ep in all_qpos)
            })
        else:
            print(f"  - Failed to load {filename}")
    
    if not all_qpos:
        print("No valid episodes loaded!")
        return None, None, None, None
    
    # Concatenate all episodes
    combined_qpos = np.concatenate(all_qpos, axis=0)
    combined_qvel = np.concatenate(all_qvel, axis=0)
    combined_action = np.concatenate(all_action, axis=0)
    
    print(f"\nCombined dataset:")
    print(f"Total episodes: {len(all_qpos)}")
    print(f"Total timesteps: {len(combined_qpos)}")
    print(f"Combined QPos shape: {combined_qpos.shape}")
    print(f"Combined Action shape: {combined_action.shape}")
    
    # Print episode breakdown
    print(f"\nEpisode breakdown:")
    for info in episode_info:
        print(f"  {info['filename']}: {info['length']} steps (indices {info['start_idx']}-{info['end_idx']-1})")
    
    return combined_qpos, combined_qvel, combined_action, all_image_dicts

# Load all episodes from the directory
data_dir = r'C:\Users\Administrator\Documents\transformer\ACT-Shaka\data\task1c'
qpos, qvel, action, image_dict_list = load_all_episodes(data_dir)

if qpos is None:
    print("Failed to load any data!")
    exit()

# Create DataFrames with joint headers
joint_headers = ['timestamp', 'b', 's', 'e', 't', 'r', 'g']

# Create timestamp column (assuming sequential timesteps across all episodes)
num_timesteps = len(qpos)
timestamps = np.arange(num_timesteps)

# Create qpos DataFrame
qpos_data = np.column_stack([timestamps, qpos])
qpos_df = pd.DataFrame(qpos_data, columns=joint_headers)

# Create action DataFrame
action_data = np.column_stack([timestamps, action])
action_df = pd.DataFrame(action_data, columns=joint_headers)

print("\nDataFrame Summary:")
print("QPos DataFrame shape:", qpos_df.shape)
print("Action DataFrame shape:", action_df.shape)
print("\nQPos DataFrame head:")
print(qpos_df.head())
print("\nAction DataFrame head:")
print(action_df.head())

# Check the data ranges for each joint
print("\nQPos data ranges:")
for col in joint_headers[1:]:  # Skip timestamp
    print(f"{col}: {qpos_df[col].min():.3f} to {qpos_df[col].max():.3f}")

print("\nAction data ranges:")
for col in joint_headers[1:]:  # Skip timestamp
    print(f"{col}: {action_df[col].min():.3f} to {action_df[col].max():.3f}")

# Optional: Plot data distribution across episodes
plt.figure(figsize=(15, 10))

joint_cols = ['b', 's', 'e', 't', 'r', 'g']
joint_names = ['Base', 'Shoulder', 'Elbow', 'Wrist1', 'Wrist2', 'Gripper']

for i, (joint, joint_name) in enumerate(zip(joint_cols, joint_names)):
    plt.subplot(2, 3, i+1)
    plt.plot(qpos_df['timestamp'], qpos_df[joint], 'b-', alpha=0.7, label=f'QPos {joint}')
    plt.plot(action_df['timestamp'], action_df[joint], 'r-', alpha=0.7, label=f'Action {joint}')
    plt.title(f'{joint_name} Joint ({joint}) - All Episodes')
    plt.xlabel('Timestep')
    plt.ylabel('Joint Value (radians)')
    plt.legend()
    plt.grid(True, alpha=0.3)

plt.suptitle('Joint Data Across All Episodes', fontsize=16)
plt.tight_layout()
plt.show()

print("\nData loading complete! Now you can proceed with preprocessing and training.")

# 1.2 Preprocessing

In [ ]:
# Complete the normalization code
joint_cols = ['b', 's', 'e', 't', 'r', 'g']

qpos_normalized = {}
action_normalized = {}

for joint in joint_cols:
    # QPos processing
    qpos_joint = qpos_df[joint].fillna(method="ffill")
    qpos_joint = np.array(qpos_joint)
    
    # Simple differences (like velocities)
    qpos_diff = np.diff(qpos_joint)
    qpos_csum_diff = qpos_diff.cumsum()
    
    # Z-score normalization of differences
    qpos_diff_normalized = (qpos_diff - np.mean(qpos_diff)) / (np.std(qpos_diff) + 1e-8)
    
    qpos_normalized[f'{joint}_diff'] = np.concatenate([[0], qpos_diff])
    qpos_normalized[f'{joint}_csum_diff'] = np.concatenate([[0], qpos_csum_diff])
    qpos_normalized[f'{joint}_diff_norm'] = np.concatenate([[0], qpos_diff_normalized])
    
    # Action processing
    action_joint = action_df[joint].fillna(method="ffill")
    action_joint = np.array(action_joint)
    
    action_diff = np.diff(action_joint)
    action_csum_diff = action_diff.cumsum()
    action_diff_normalized = (action_diff - np.mean(action_diff)) / (np.std(action_diff) + 1e-8)
    
    action_normalized[f'{joint}_diff'] = np.concatenate([[0], action_diff])
    action_normalized[f'{joint}_csum_diff'] = np.concatenate([[0], action_csum_diff])
    action_normalized[f'{joint}_diff_norm'] = np.concatenate([[0], action_diff_normalized])

# Create DataFrames from normalized data
qpos_norm_df = pd.DataFrame(qpos_normalized)
qpos_norm_df.insert(0, 'timestamp', qpos_df['timestamp'])

action_norm_df = pd.DataFrame(action_normalized)
action_norm_df.insert(0, 'timestamp', action_df['timestamp'])

print("Normalization completed!")
print(f"QPos normalized shape: {qpos_norm_df.shape}")
print(f"Action normalized shape: {action_norm_df.shape}")

In [ ]:
qpos_norm_df.head()

In [ ]:
# Plot raw and cumulative sum for base joint ('b') of both dataframes
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# QPos plots
axes[0, 0].plot(qpos_df['timestamp'], qpos_df['b'], 'b-', linewidth=2, label='Raw QPos')
axes[0, 0].set_title('Base Joint - Raw QPos')
axes[0, 0].set_xlabel('Timestamp')
axes[0, 0].set_ylabel('Joint Position (radians)')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()

axes[0, 1].plot(qpos_norm_df['timestamp'], qpos_norm_df['b_csum_diff'], 'g-', linewidth=2, label='Cumulative Sum Diff')
axes[0, 1].set_title('Base Joint - QPos Cumulative Sum of Differences')
axes[0, 1].set_xlabel('Timestamp')
axes[0, 1].set_ylabel('Cumulative Difference')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()

# Action plots
axes[1, 0].plot(action_df['timestamp'], action_df['b'], 'r-', linewidth=2, label='Raw Action')
axes[1, 0].set_title('Base Joint - Raw Action')
axes[1, 0].set_xlabel('Timestamp')
axes[1, 0].set_ylabel('Action Value (radians)')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend()

axes[1, 1].plot(action_norm_df['timestamp'], action_norm_df['b_csum_diff'], 'm-', linewidth=2, label='Cumulative Sum Diff')
axes[1, 1].set_title('Base Joint - Action Cumulative Sum of Differences')
axes[1, 1].set_xlabel('Timestamp')
axes[1, 1].set_ylabel('Cumulative Difference')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend()

plt.tight_layout()
plt.show()

# Additional comparison plot - overlay raw vs normalized differences
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# QPos comparison
axes[0].plot(qpos_df['timestamp'], qpos_df['b'], 'b-', alpha=0.7, label='Raw QPos')
axes[0].plot(qpos_norm_df['timestamp'], qpos_norm_df['b_diff'], 'g-', alpha=0.7, label='Differences')
axes[0].plot(qpos_norm_df['timestamp'], qpos_norm_df['b_diff_norm'], 'orange', alpha=0.7, label='Normalized Differences')
axes[0].set_title('Base Joint - QPos Transformations')
axes[0].set_xlabel('Timestamp')
axes[0].set_ylabel('Value')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Action comparison
axes[1].plot(action_df['timestamp'], action_df['b'], 'r-', alpha=0.7, label='Raw Action')
axes[1].plot(action_norm_df['timestamp'], action_norm_df['b_diff'], 'purple', alpha=0.7, label='Differences')
axes[1].plot(action_norm_df['timestamp'], action_norm_df['b_diff_norm'], 'cyan', alpha=0.7, label='Normalized Differences')
axes[1].set_title('Base Joint - Action Transformations')
axes[1].set_xlabel('Timestamp')
axes[1].set_ylabel('Value')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

# Print some statistics
print("\nBase Joint Statistics:")
print("=" * 50)
print("QPos:")
print(f"  Raw range: {qpos_df['b'].min():.3f} to {qpos_df['b'].max():.3f}")
print(f"  Diff range: {qpos_norm_df['b_diff'].min():.3f} to {qpos_norm_df['b_diff'].max():.3f}")
print(f"  Cumsum range: {qpos_norm_df['b_csum_diff'].min():.3f} to {qpos_norm_df['b_csum_diff'].max():.3f}")

print("\nAction:")
print(f"  Raw range: {action_df['b'].min():.3f} to {action_df['b'].max():.3f}")
print(f"  Diff range: {action_norm_df['b_diff'].min():.3f} to {action_norm_df['b_diff'].max():.3f}")
print(f"  Cumsum range: {action_norm_df['b_csum_diff'].min():.3f} to {action_norm_df['b_csum_diff'].max():.3f}")

In [ ]:
# Plot raw and cumulative sum for all joints of both dataframes
joint_cols = ['b', 's', 'e', 't', 'r', 'g']
joint_names = ['Base', 'Shoulder', 'Elbow', 'Wrist1', 'Wrist2', 'Gripper']

# Create subplots for all joints - 2x3 grid for 6 joints
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()  # Flatten for easier indexing

for i, (joint, joint_name) in enumerate(zip(joint_cols, joint_names)):
    # Plot QPos and Action on the same subplot for comparison
    axes[i].plot(qpos_df['timestamp'], qpos_df[joint], 'b-', linewidth=2, label=f'QPos {joint}', alpha=0.8)
    axes[i].plot(action_df['timestamp'], action_df[joint], 'r-', linewidth=2, label=f'Action {joint}', alpha=0.8)
    axes[i].set_title(f'{joint_name} Joint ({joint}) - Raw Values Comparison')
    axes[i].set_xlabel('Timestamp')
    axes[i].set_ylabel('Joint Value (radians)')
    axes[i].grid(True, alpha=0.3)
    axes[i].legend()

plt.tight_layout()
plt.show()

# Create cumulative sum comparison plots
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, (joint, joint_name) in enumerate(zip(joint_cols, joint_names)):
    # Plot cumulative sums for comparison
    axes[i].plot(qpos_norm_df['timestamp'], qpos_norm_df[f'{joint}_csum_diff'], 'g-', linewidth=2, label=f'QPos CumSum {joint}', alpha=0.8)
    axes[i].plot(action_norm_df['timestamp'], action_norm_df[f'{joint}_csum_diff'], 'm-', linewidth=2, label=f'Action CumSum {joint}', alpha=0.8)
    axes[i].set_title(f'{joint_name} Joint ({joint}) - Cumulative Sum of Differences')
    axes[i].set_xlabel('Timestamp')
    axes[i].set_ylabel('Cumulative Difference')
    axes[i].grid(True, alpha=0.3)
    axes[i].legend()

plt.tight_layout()
plt.show()

# Detailed transformations plot for all joints
fig, axes = plt.subplots(3, 2, figsize=(15, 18))

for i, (joint, joint_name) in enumerate(zip(joint_cols, joint_names)):
    row = i // 2
    col = i % 2
    
    # Plot raw, differences, and normalized differences
    axes[row, col].plot(qpos_df['timestamp'], qpos_df[joint], 'b-', alpha=0.7, label=f'Raw QPos {joint}')
    axes[row, col].plot(action_df['timestamp'], action_df[joint], 'r-', alpha=0.7, label=f'Raw Action {joint}')
    axes[row, col].plot(qpos_norm_df['timestamp'], qpos_norm_df[f'{joint}_diff'], 'g--', alpha=0.7, label=f'QPos Diff {joint}')
    axes[row, col].plot(action_norm_df['timestamp'], action_norm_df[f'{joint}_diff'], 'purple', alpha=0.7, label=f'Action Diff {joint}')
    axes[row, col].set_title(f'{joint_name} Joint ({joint}) - All Transformations')
    axes[row, col].set_xlabel('Timestamp')
    axes[row, col].set_ylabel('Value')
    axes[row, col].grid(True, alpha=0.3)
    axes[row, col].legend()

plt.tight_layout()
plt.show()

# Print comprehensive statistics for all joints
print("\nAll Joints Statistics:")
print("=" * 80)

for joint, joint_name in zip(joint_cols, joint_names):
    print(f"\n{joint_name} Joint ({joint}):")
    print("-" * 40)
    print("QPos:")
    print(f"  Raw range: {qpos_df[joint].min():.3f} to {qpos_df[joint].max():.3f}")
    print(f"  Diff range: {qpos_norm_df[f'{joint}_diff'].min():.3f} to {qpos_norm_df[f'{joint}_diff'].max():.3f}")
    print(f"  Cumsum range: {qpos_norm_df[f'{joint}_csum_diff'].min():.3f} to {qpos_norm_df[f'{joint}_csum_diff'].max():.3f}")
    
    print("Action:")
    print(f"  Raw range: {action_df[joint].min():.3f} to {action_df[joint].max():.3f}")
    print(f"  Diff range: {action_norm_df[f'{joint}_diff'].min():.3f} to {action_norm_df[f'{joint}_diff'].max():.3f}")
    print(f"  Cumsum range: {action_norm_df[f'{joint}_csum_diff'].min():.3f} to {action_norm_df[f'{joint}_csum_diff'].max():.3f}")
    
    # Calculate correlation between qpos and action
    correlation = np.corrcoef(qpos_df[joint], action_df[joint])[0, 1]
    print(f"  QPos-Action correlation: {correlation:.3f}")

# Additional analysis: Check temporal alignment
print("\n" + "=" * 80)
print("TEMPORAL ALIGNMENT ANALYSIS:")
print("=" * 80)

for joint, joint_name in zip(joint_cols, joint_names):
    # Calculate cross-correlation to find optimal lag
    qpos_values = qpos_df[joint].values
    action_values = action_df[joint].values
    
    # Normalize for cross-correlation
    qpos_norm = (qpos_values - np.mean(qpos_values)) / np.std(qpos_values)
    action_norm = (action_values - np.mean(action_values)) / np.std(action_values)
    
    # Calculate cross-correlation
    cross_corr = np.correlate(qpos_norm, action_norm, mode='full')
    lags = np.arange(-len(action_norm) + 1, len(qpos_norm))
    
    # Find the lag with maximum correlation
    max_corr_idx = np.argmax(np.abs(cross_corr))
    optimal_lag = lags[max_corr_idx]
    max_correlation = cross_corr[max_corr_idx]
    
    print(f"{joint_name} ({joint}):")
    print(f"  Optimal lag: {optimal_lag} timesteps")
    print(f"  Max cross-correlation: {max_correlation:.3f}")
    if optimal_lag > 0:
        print(f"  → QPos leads Action by {optimal_lag} steps")
    elif optimal_lag < 0:
        print(f"  → Action leads QPos by {abs(optimal_lag)} steps")
    else:
        print(f"  → QPos and Action are synchronized")
    print()

# 2. Model Definition

## 2.1 Positional Encoding Layer

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:x.size(0), :]

## 2.2. Transformer

In [ ]:
class RobotTransformer(nn.Module):
    def __init__(self, 
                 input_dim=6,      # 6 joint positions (qpos)
                 output_dim=6,     # 6 joint actions
                 d_model=256,      # Embedding dimension
                 nhead=8,          # Number of attention heads
                 num_layers=6,     # Number of transformer layers
                 dropout=0.1):
        super(RobotTransformer, self).__init__()
        
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.d_model = d_model
        
        # Input embedding: map 6 joint positions to d_model dimensions
        self.input_embedding = nn.Linear(input_dim, d_model)
        
        # Positional encoding
        self.pos_encoder = PositionalEncoding(d_model)
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=False  # [seq_len, batch, features]
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers)
        
        # Output projection: map d_model to 6 joint actions
        self.output_projection = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, output_dim)
        )
        
        # Causal mask for autoregressive prediction
        self.register_buffer('causal_mask', None)
        
        self.init_weights()
    #
    def init_weights(self):
        initrange = 0.1
        self.input_embedding.weight.data.uniform_(-initrange, initrange)
        self.output_projection[0].weight.data.uniform_(-initrange, initrange)
        self.output_projection[3].weight.data.uniform_(-initrange, initrange)
    
    def _generate_square_subsequent_mask(self, sz):
        """Generate causal mask to prevent looking at future timesteps"""
        mask = torch.triu(torch.ones(sz, sz), diagonal=1)
        mask = mask.masked_fill(mask == 1, float('-inf'))
        return mask
    
    def forward(self, src):
        """
        Args:
            src: [seq_len, batch, input_dim] - sequence of joint positions
        Returns:
            output: [seq_len, batch, output_dim] - predicted joint actions
        """
        seq_len = src.size(0)
        
        # Generate or reuse causal mask
        if self.causal_mask is None or self.causal_mask.size(0) != seq_len:
            self.causal_mask = self._generate_square_subsequent_mask(seq_len).to(src.device)
        
        # Input embedding: [seq_len, batch, input_dim] -> [seq_len, batch, d_model]
        src = self.input_embedding(src)
        
        # Add positional encoding
        src = self.pos_encoder(src)
        
        # Transformer encoding with causal mask
        transformer_output = self.transformer_encoder(src, self.causal_mask)
        
        # Project to action space: [seq_len, batch, d_model] -> [seq_len, batch, output_dim]
        actions = self.output_projection(transformer_output)
        
        return actions

In [ ]:
# Initialize transformer model with 250-dimensional embeddings
model = RobotTransformer().to(device)

# 3. Utils Functions

## 3.1. Create window sequence

In [ ]:
def create_robot_sequences(qpos_data, action_data, sequence_length):
    """
    Create input-output sequences for robot training
    Args:
        qpos_data: [timesteps, 6] - joint positions
        action_data: [timesteps, 6] - joint actions
        sequence_length: length of input sequence
    Returns:
        sequences: list of (input_seq, target_seq) tuples
    """
    sequences = []
    
    for i in range(len(qpos_data) - sequence_length):
        # Input: sequence of joint positions
        input_seq = qpos_data[i:i+sequence_length]  # [seq_len, 6]
        
        # Target: corresponding sequence of actions
        target_seq = action_data[i:i+sequence_length]  # [seq_len, 6]
        
        sequences.append((input_seq, target_seq))
    
    return sequences

## 3.2 Split data
 to training and test sets

In [ ]:
def get_robot_data(qpos_df, action_df, sequence_length, train_split=0.8):
    """
    Prepare robot data for training
    """
    # Extract joint columns (skip timestamp)
    joint_cols = ['b', 's', 'e', 't', 'r', 'g']
    
    qpos_data = qpos_df[joint_cols].values  # [timesteps, 6]
    action_data = action_df[joint_cols].values  # [timesteps, 6]
    
    # Normalize data (important for stable training)
    qpos_mean = qpos_data.mean(axis=0)
    qpos_std = qpos_data.std(axis=0) + 1e-8
    qpos_normalized = (qpos_data - qpos_mean) / qpos_std
    
    action_mean = action_data.mean(axis=0)
    action_std = action_data.std(axis=0) + 1e-8
    action_normalized = (action_data - action_mean) / action_std
    
    # Create sequences
    sequences = create_robot_sequences(qpos_normalized, action_normalized, sequence_length)
    
    # Split into train/test
    split_idx = int(len(sequences) * train_split)
    train_sequences = sequences[:split_idx]
    test_sequences = sequences[split_idx:]
    
    # Convert to tensors
    train_data = [(torch.FloatTensor(seq[0]), torch.FloatTensor(seq[1])) 
                  for seq in train_sequences]
    test_data = [(torch.FloatTensor(seq[0]), torch.FloatTensor(seq[1])) 
                 for seq in test_sequences]
    
    # Store normalization parameters for later use
    normalization_params = {
        'qpos_mean': qpos_mean,
        'qpos_std': qpos_std,
        'action_mean': action_mean,
        'action_std': action_std
    }
    
    return train_data, test_data, normalization_params

In [ ]:
def get_robot_data_raw(qpos_df, action_df, sequence_length, train_split=0.8):
    """Use raw joint data instead of normalized differences"""
    joint_cols = ['b', 's', 'e', 't', 'r', 'g']
    
    # Use RAW data directly
    qpos_data = qpos_df[joint_cols].values
    action_data = action_df[joint_cols].values
    
    # Simple min-max normalization instead of z-score
    qpos_min, qpos_max = qpos_data.min(axis=0), qpos_data.max(axis=0)
    qpos_normalized = (qpos_data - qpos_min) / (qpos_max - qpos_min + 1e-8)
    
    action_min, action_max = action_data.min(axis=0), action_data.max(axis=0)
    action_normalized = (action_data - action_min) / (action_max - action_min + 1e-8)
    
    
    sequences = create_robot_sequences(qpos_normalized, action_normalized, sequence_length)
    
    split_idx = int(len(sequences) * train_split)
    train_sequences = sequences[:split_idx]
    test_sequences = sequences[split_idx:]
    
    train_data = [(torch.FloatTensor(seq[0]), torch.FloatTensor(seq[1])) 
                  for seq in train_sequences]
    test_data = [(torch.FloatTensor(seq[0]), torch.FloatTensor(seq[1])) 
                 for seq in test_sequences]
    
    normalization_params = {
        'qpos_min': qpos_min, 'qpos_max': qpos_max,
        'action_min': action_min, 'action_max': action_max
    }
    
    return train_data, test_data, normalization_params


## 3.3. Split into training batches

In [ ]:
# chunking + stacking rotates the data from batch-first to time-first format.
'''
what the key line:" input = torch.stack(torch.stack([item[0] for item in data]).chunk(input_window, 1))" does: 
# Input sequences in batch-first format [batch, sequence]:
[[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0],   # Sequence 1, base joint
 [2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0]]  # Sequence 2, shoulder joint
 
# Rearranged to time-first format [time, batch]:
# Timestep 0: [1.0, 2.0]  - What both sequences have at time 0
# Timestep 1: [2.0, 3.0]  - What both sequences have at time 1
# Timestep 2: [3.0, 4.0]  - What both sequences have at time 2

'''
def get_robot_batch(sequences, start_idx, batch_size):
    """
    Create batches for robot data
    Args:
        sequences: list of (input_seq, target_seq) tuples
        start_idx: starting index
        batch_size: batch size
    Returns:
        inputs: [seq_len, batch, 6] - batched input sequences
        targets: [seq_len, batch, 6] - batched target sequences
    """
    end_idx = min(start_idx + batch_size, len(sequences))
    batch_sequences = sequences[start_idx:end_idx]
    
    # Stack sequences into batches
    inputs = torch.stack([seq[0] for seq in batch_sequences], dim=1)  # [seq_len, batch, 6]
    targets = torch.stack([seq[1] for seq in batch_sequences], dim=1)  # [seq_len, batch, 6]
    
    return inputs, targets

## 3.4. training function and training configuration

In [ ]:
# Loss function: Mean Squared Error for regression
# Formula: MSE = (1/n) * Σ(y_pred - y_true)²
criterion = nn.MSELoss()

# Learning rate and training epochs
lr = 0.00005     # Small learning rate for stable training
epochs = 1000     # Number of complete passes through training data

# Adam optimizer with weight decay for regularization
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
# Learning rate scheduler: reduces LR by factor of 0.95 each epoch
# Formula: new_lr = current_lr * 0.95
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, 1.0, gamma=0.95)


# 4 Train phase

In [ ]:
def train_robot_model(model, train_data, val_data, device, epochs=100):
    """
    Train the robot transformer model
    """
    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
    
    batch_size = 32
    train_losses = []
    val_losses = []
    
    print(f"Training robot transformer on {device}")
    print(f"Train sequences: {len(train_data)}, Val sequences: {len(val_data)}")
    
    for epoch in range(epochs):
        # Training
        model.train()
        total_train_loss = 0.0
        num_batches = 0
        
        for i in range(0, len(train_data), batch_size):
            inputs, targets = get_robot_batch(train_data, i, batch_size)
            inputs, targets = inputs.to(device), targets.to(device)
            
            optimizer.zero_grad()
            
            # Forward pass
            predictions = model(inputs)  # [seq_len, batch, 6]
            
            # Calculate loss
            loss = criterion(predictions, targets)
            
            # Backward pass
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            total_train_loss += loss.item()
            num_batches += 1
        
        avg_train_loss = total_train_loss / num_batches
        train_losses.append(avg_train_loss)
        
        # Validation
        model.eval()
        total_val_loss = 0.0
        num_val_batches = 0
        
        with torch.no_grad():
            for i in range(0, len(val_data), batch_size):
                inputs, targets = get_robot_batch(val_data, i, batch_size)
                inputs, targets = inputs.to(device), targets.to(device)
                
                predictions = model(inputs)
                loss = criterion(predictions, targets)
                
                total_val_loss += loss.item()
                num_val_batches += 1
        
        avg_val_loss = total_val_loss / num_val_batches
        val_losses.append(avg_val_loss)
        
        # Learning rate scheduling
        scheduler.step(avg_val_loss)
        
        # Print progress
        if epoch % 10 == 0:
            print(f"Epoch {epoch:3d} | Train Loss: {avg_train_loss:.6f} | "
                  f"Val Loss: {avg_val_loss:.6f} | LR: {optimizer.param_groups[0]['lr']:.2e}")
    
    return train_losses, val_losses

In [ ]:
# Initialize the robot transformer
print(f"Using device: {device}")

# Model parameters
model = RobotTransformer(
    input_dim=6,      # 6 joint positions
    output_dim=6,     # 6 joint actions  
    d_model=256,      # Embedding dimension
    nhead=8,          # Attention heads
    num_layers=4,     # Transformer layers
    dropout=0.1
).to(device)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# Prepare data
sequence_length = 100
train_data, val_data, norm_params = get_robot_data_raw(
    qpos_df, action_df, 
    sequence_length=sequence_length,
    train_split=0.8
)

print(f"Training sequences: {len(train_data)}")
print(f"Validation sequences: {len(val_data)}")
print(f"Sequence length: {sequence_length}")

# Train the model
train_losses, val_losses = train_robot_model(
    model, train_data, val_data, device, epochs=epochs
)

# Plot training progress
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training Progress')
plt.legend()
plt.yscale('log')

plt.subplot(1, 2, 2)
plt.plot(train_losses[-50:], label='Train Loss (Last 50)')
plt.plot(val_losses[-50:], label='Val Loss (Last 50)')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Recent Training Progress')
plt.legend()

plt.tight_layout()
plt.show()

## 3.5 evaluation function

In [ ]:
def evaluate_robot_model(model, val_data, norm_params, device, num_steps=10, num_sequences=5):
    """
    Evaluate robot model with multi-step prediction and visualization
    Args:
        model: trained transformer model
        val_data: validation sequences
        norm_params: normalization parameters for denormalizing predictions
        device: torch device
        num_steps: number of consecutive steps to predict
        num_sequences: number of validation sequences to evaluate
    Returns:
        avg_loss: average validation loss
        predictions: denormalized predictions
        actuals: denormalized actual values
    """
    model.eval()
    criterion = nn.MSELoss()
    
    joint_names = ['Base', 'Shoulder', 'Elbow', 'Wrist1', 'Wrist2', 'Gripper']
    joint_cols = ['b', 's', 'e', 't', 'r', 'g']
    
    # Storage for results
    all_predictions = []
    all_actuals = []
    all_losses = []
    
    with torch.no_grad():
        # Select random sequences for detailed evaluation
        selected_indices = np.random.choice(len(val_data), 
                                          size=min(num_sequences, len(val_data)), 
                                          replace=False)
        
        print(f"Evaluating {len(selected_indices)} sequences with {num_steps} consecutive steps each")
        
        for seq_idx in selected_indices:
            # Get one sequence
            input_seq, target_seq = val_data[seq_idx]
            
            # Prepare for multi-step prediction
            current_input = input_seq.unsqueeze(1).to(device)  # [seq_len, 1, 6]
            sequence_predictions = []
            sequence_actuals = []
            
            # Multi-step prediction
            for step in range(num_steps):
                if step < len(target_seq):
                    # Predict next action
                    prediction = model(current_input)  # [seq_len, 1, 6], [100 steps , one batch, 6 joint states]
                    
                    # Get the last timestep prediction
                    step_prediction = prediction[-1, 0, :].cpu()  # [6 joint states]
                    step_actual = target_seq[step]  # [6 joint states]

                    sequence_predictions.append(step_prediction)
                    sequence_actuals.append(step_actual)
                    
                    # For next step, append the actual target (teacher forcing for evaluation)
                    # Alternative: use prediction for autoregressive evaluation
                    next_input = step_actual.unsqueeze(0).unsqueeze(1).to(device)  # [1, 1, 6]
                    current_input = torch.cat([current_input[1:], next_input], dim=0)
            
            # Convert to tensors
            if sequence_predictions:
                seq_pred = torch.stack(sequence_predictions)  # [num_steps, 6]
                seq_actual = torch.stack(sequence_actuals)    # [num_steps, 6]
                
                all_predictions.append(seq_pred)
                all_actuals.append(seq_actual)
                
                # Calculate loss for this sequence
                seq_loss = criterion(seq_pred, seq_actual).item()
                all_losses.append(seq_loss)
    
    # Calculate average loss
    avg_loss = np.mean(all_losses) if all_losses else float('inf')
    
    # REVISED: Denormalize predictions and actuals for min-max normalization
    # Check if using min-max (get_robot_data_raw) or z-score normalization
    if 'action_min' in norm_params and 'action_max' in norm_params:
        # Min-max denormalization: value = normalized * (max - min) + min
        action_min = torch.FloatTensor(norm_params['action_min'])
        action_max = torch.FloatTensor(norm_params['action_max'])
        action_range = action_max - action_min
        
        denorm_predictions = []
        denorm_actuals = []
        
        for pred, actual in zip(all_predictions, all_actuals):
            # Denormalize: value = normalized * (max - min) + min
            denorm_pred = pred * action_range + action_min
            denorm_actual = actual * action_range + action_min
            
            # Convert to numpy for plotting
            denorm_predictions.append(denorm_pred.numpy())
            denorm_actuals.append(denorm_actual.numpy())
    
    elif 'action_mean' in norm_params and 'action_std' in norm_params:
        # Z-score denormalization: value = normalized * std + mean
        action_mean = torch.FloatTensor(norm_params['action_mean'])
        action_std = torch.FloatTensor(norm_params['action_std'])
        
        denorm_predictions = []
        denorm_actuals = []
        
        for pred, actual in zip(all_predictions, all_actuals):
            # Denormalize: value = normalized * std + mean
            denorm_pred = pred * action_std + action_mean
            denorm_actual = actual * action_std + action_mean
            
            # Convert to numpy for plotting
            denorm_predictions.append(denorm_pred.numpy())
            denorm_actuals.append(denorm_actual.numpy())
    
    else:
        # Fallback: use normalized values if no proper normalization params found
        print("Warning: No proper normalization parameters found. Using normalized values.")
        denorm_predictions = [pred.numpy() for pred in all_predictions]
        denorm_actuals = [actual.numpy() for actual in all_actuals]
    
    return avg_loss, denorm_predictions, denorm_actuals, joint_names, joint_cols


def plot_evaluation_results(predictions, actuals, joint_names, joint_cols, num_steps=None):
    """
    Create comprehensive plots comparing predicted vs actual joint values
    """
    num_sequences = len(predictions)
    
    # Plot 1: All joints for first sequence
    if num_sequences > 0:
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        axes = axes.flatten()
        
        pred_seq = predictions[0]  # [num_steps, 6] - numpy array
        actual_seq = actuals[0]    # [num_steps, 6] - numpy array
        
        # Auto-detect number of steps if not provided
        if num_steps is None:
            num_steps = min(len(pred_seq), len(actual_seq))
        else:
            num_steps = min(num_steps, len(pred_seq), len(actual_seq))
        
        for i, (joint_name, joint_col) in enumerate(zip(joint_names, joint_cols)):
            time_steps = range(num_steps)
            
            axes[i].plot(time_steps, actual_seq[:num_steps, i], 'b-o', linewidth=2, 
                        label=f'Actual {joint_col}', alpha=0.8)
            axes[i].plot(time_steps, pred_seq[:num_steps, i], 'r--s', linewidth=2, 
                        label=f'Predicted {joint_col}', alpha=0.8)
            
            axes[i].set_title(f'{joint_name} Joint ({joint_col})')
            axes[i].set_xlabel('Time Step')
            axes[i].set_ylabel('Joint Value (radians)')
            axes[i].legend()
            axes[i].grid(True, alpha=0.3)
            
            # Calculate and display error metrics using numpy
            mse = np.mean((actual_seq[:num_steps, i] - pred_seq[:num_steps, i])**2)
            mae = np.mean(np.abs(actual_seq[:num_steps, i] - pred_seq[:num_steps, i]))
            axes[i].text(0.05, 0.95, f'MSE: {mse:.4f}\nMAE: {mae:.4f}', 
                        transform=axes[i].transAxes, verticalalignment='top',
                        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
        
        plt.suptitle(f'Joint Predictions vs Actuals - Sequence 1 ({num_steps} steps)', fontsize=16)
        plt.tight_layout()
        plt.show()
    
    # Plot 2: Error analysis across all sequences
    if num_sequences > 1:
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        axes = axes.flatten()
        
        # Auto-detect max steps across all sequences
        if num_steps is None:
            max_steps = max(len(pred) for pred in predictions)
        else:
            max_steps = num_steps
        
        for i, (joint_name, joint_col) in enumerate(zip(joint_names, joint_cols)):
            errors_per_step = []
            
            for step in range(max_steps):
                step_errors = []
                for seq_idx in range(num_sequences):
                    if step < len(predictions[seq_idx]) and step < len(actuals[seq_idx]):
                        error = abs(actuals[seq_idx][step, i] - predictions[seq_idx][step, i])
                        step_errors.append(error)
                
                if step_errors:
                    errors_per_step.append(step_errors)
            
            # Box plot of errors across sequences for each time step
            if errors_per_step:
                axes[i].boxplot(errors_per_step, positions=range(1, len(errors_per_step)+1))
                axes[i].set_title(f'{joint_name} Joint ({joint_col}) - Error Distribution')
                axes[i].set_xlabel('Prediction Step')
                axes[i].set_ylabel('Absolute Error (radians)')
                axes[i].grid(True, alpha=0.3)
        
        plt.suptitle(f'Prediction Error Analysis Across {num_sequences} Sequences', fontsize=16)
        plt.tight_layout()
        plt.show()
    
    # Plot 3: Correlation analysis
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    
    for i, (joint_name, joint_col) in enumerate(zip(joint_names, joint_cols)):
        all_actual_vals = []
        all_pred_vals = []
        
        for seq_idx in range(num_sequences):
            # Use all available steps for correlation analysis
            seq_length = min(len(predictions[seq_idx]), len(actuals[seq_idx]))
            all_actual_vals.extend(actuals[seq_idx][:seq_length, i].tolist())
            all_pred_vals.extend(predictions[seq_idx][:seq_length, i].tolist())
        
        if all_actual_vals and all_pred_vals:
            axes[i].scatter(all_actual_vals, all_pred_vals, alpha=0.6, s=20)
            
            # Perfect prediction line
            min_val = min(min(all_actual_vals), min(all_pred_vals))
            max_val = max(max(all_actual_vals), max(all_pred_vals))
            axes[i].plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.8, label='Perfect Prediction')
            
            # Calculate correlation
            correlation = np.corrcoef(all_actual_vals, all_pred_vals)[0, 1]
            
            axes[i].set_title(f'{joint_name} Joint ({joint_col})\nCorr: {correlation:.3f}')
            axes[i].set_xlabel('Actual Values (radians)')
            axes[i].set_ylabel('Predicted Values (radians)')
            axes[i].legend()
            axes[i].grid(True, alpha=0.3)
    
    plt.suptitle('Predicted vs Actual Correlation Analysis', fontsize=16)
    plt.tight_layout()
    plt.show()

def comprehensive_model_evaluation(model, val_data, norm_params, device):
    """
    Complete evaluation pipeline
    """
    print("Starting comprehensive model evaluation...")
    print("="*60)
    
    # Evaluate model
    avg_loss, predictions, actuals, joint_names, joint_cols = evaluate_robot_model(
        model, val_data, norm_params, device, num_steps=100, num_sequences=5
    )
    
    print(f"Average Validation Loss: {avg_loss:.6f}")
    
    # Calculate overall statistics
    if predictions and actuals:
        all_errors = []
        joint_errors = {joint: [] for joint in joint_cols}
        
        for pred_seq, actual_seq in zip(predictions, actuals):
            seq_length = min(len(pred_seq), len(actual_seq))
            for step in range(seq_length):
                for joint_idx, joint in enumerate(joint_cols):
                    error = abs(actual_seq[step, joint_idx] - pred_seq[step, joint_idx])
                    all_errors.append(error)
                    joint_errors[joint].append(error)
        
        print(f"\nOverall Statistics:")
        print(f"Mean Absolute Error: {np.mean(all_errors):.4f} radians")
        print(f"Max Absolute Error: {np.max(all_errors):.4f} radians")
        print(f"Std Absolute Error: {np.std(all_errors):.4f} radians")
        
        print(f"\nPer-Joint Statistics:")
        for joint in joint_cols:
            if joint_errors[joint]:
                print(f"Joint {joint}: MAE = {np.mean(joint_errors[joint]):.4f}, "
                      f"Max = {np.max(joint_errors[joint]):.4f}")
    
    # Create plots - let the function auto-detect the number of steps
    if predictions and actuals:
        plot_evaluation_results(predictions, actuals, joint_names, joint_cols)
    
    return avg_loss, predictions, actuals

In [ ]:
# After training your model
avg_loss, predictions, actuals = comprehensive_model_evaluation(
    model, val_data, norm_params, device
)